# Test Case 1 — Minimal RCA Orchestrator Smoke Test

## Purpose

This notebook is a **minimal end-to-end dry run** of the full RCA orchestrator pipeline.
It verifies that all pipeline stages wire together correctly without requiring a live Neo4j
database or an LLM service. All intermediate artifacts are supplied as pre-built JSON fixtures,
so the orchestrator's Stage G (finalize manifest, schema validation, RCA card generation) runs
on controlled inputs and produces a deterministic result.

## Scenario

| Field | Value |
|---|---|
| **Event ID** | `E2026-01-23-001` |
| **Asset** | `PUMP_A_01` (centrifugal pump) |
| **Failure mode under test** | `FM_BEARING_WEAR` — progressive bearing wear on component `CMP_BRG` |
| **Telemetry signal** | `S1_VIB` — vibration sensor showing an anomaly window 10:00–10:30 UTC |
| **Supporting document** | `CR:DEMO:0001` — condition report noting "Bearing wear suspected" |
| **Expected primary hypothesis** | Bearing wear, `composite_score ≈ 0.78` |

This is deliberately a **simple, single-candidate scenario** designed to validate orchestrator
plumbing, not to exercise the full discriminating power of the scoring engine.

## What this test covers

- Artifact schema validation (input bundle and output bundle) via `RCAArtifactValidator`
- Orchestrator construction via `build_dev_orchestrator` (schema loading, validator wiring, synthesizer)
- Backward-compatibility normalization for fixtures missing the `telemetry` score dimension
  or `temporal_evidence` fields introduced in later schema versions
- Stub TSKR pattern generation when `tskr_patterns.json` is absent from the fixture directory
- RCA card generation via the rule-validated fallback synthesizer
- `run_manifest` population (artifact presence flags, validation status, review hooks)

## What this test does NOT cover

- **Live Neo4j queries** — the KG context is fully pre-built; the Neo4j client is instantiated
  but no queries reach the database
- **LLM synthesis** — Ollama is not required; the fallback rule-based synthesizer is used
- **Evidence retrieval** — the `evidence_bundle` fixture is pre-built; no vector store is queried
- **Multi-candidate discrimination** — only one failure mode candidate is present in the fixture

## Fixture files

All fixtures live under `fixtures/case_001_bearing_wear/`.

| File | Required | Description |
|---|---|---|
| `event.json` | Yes | Abnormal event descriptor (asset, severity, timestamp, symptom signature) |
| `telemetry_summary.json` | Yes | Aggregated sensor anomaly summary for `PUMP_A_01` |
| `kg_context.json` | Yes | KG neighborhood: components, failure modes, past events, documents |
| `causality_candidates.json` | Yes | Pre-scored candidate list from the causality engine |
| `evidence_bundle.json` | Yes | Retrieved evidence snippets mapped to candidates |
| `operational_context.json` | No | Operational state context (load, mode, alarms) |
| `pm_compliance.json` | No | Preventive maintenance compliance record |
| `rca_card.json` | No | Pre-existing RCA card seed (unused in this run) |
| `tskr_patterns.json` | No | TSKR temporal scoring patterns; auto-generated as a stub if absent |

## Expected outputs

| Field | Expected value |
|---|---|
| `input_validation.ok` | `true` |
| `output_validation.ok` | `true` |
| `rca_card.primary_hypothesis.candidate_id` | `FM::FM_BEARING_WEAR` |
| `rca_card.primary_hypothesis.composite_score` | `0.78` |
| `rca_card.validation_status.schema_valid` | `true` |
| `run_manifest.review_hooks.requires_human_review` | `true` |
| `run_manifest.review_hooks.writeback_ready` | `true` |

## Pipeline architecture reference

The orchestrator executes these stages in order:

```
Stage A — Build run context (run_id, timestamps, config)
Stage B — KG context builder  ← pre-supplied from fixture
Stage C — TSKR temporal scorer ← pre-supplied (or stub generated)
Stage D — Causality engine     ← pre-supplied from fixture
Stage E — Evidence retriever   ← pre-supplied from fixture
Stage F — RCA synthesizer      ← runs live (fallback rule-based)
Stage G — Finalize manifest    ← runs live (validation + writeback check)
```

Stages B–E are bypassed by passing pre-built artifacts directly to `orchestrator.run()`.
Only Stages F and G execute with live logic.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import sys
from typing import Any, Dict, Optional

rca_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if rca_root not in sys.path:
    sys.path.insert(0, rca_root)

dackar_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if dackar_root not in sys.path:
    sys.path.insert(0, dackar_root)

from kg.py2neo_workflow import Py2Neo
from orchestrators.rca_reasoning_orchestrator import build_dev_orchestrator

In [ ]:
EXTRACT_DIR = Path("fixtures/case_001_bearing_wear")
OUTPUT_DIR = Path("./rca_runs_notebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_DIR = Path("../../schemas")

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

VALIDATOR_MODE = "compat"
STOP_ON_VALIDATION_ERROR = False


## Utility functions

### Fixture loaders
- `load_json(path)` — load a required JSON file; raises `FileNotFoundError` if absent
- `maybe_load_json(path)` — load an optional JSON file; returns `None` if absent

### Backward-compatibility normalizers

The fixture files were originally created for an earlier schema version.
These functions bring them up to the current schema without modifying the fixture files:

- **`normalize_event`** — adds `event_id` from legacy `id` field if missing
- **`normalize_kg_context`** — backfills `asset_id` from `seed_context.asset_ids` and
  generates a `subgraph_id` if either is absent
- **`normalize_candidates`** — adds the `telemetry` score dimension (introduced after the
  fixture was built), backfills `score_rationale` and `temporal_evidence` fields
- **`derive_tskr_patterns`** — generates a minimal TSKR stub from the failure modes in
  `kg_context` when `tskr_patterns.json` is not present in the fixture directory

### Helper
- `safe_get(d, *keys, default)` — safe nested dict lookup; returns `default` on any missing key

In [ ]:
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def maybe_load_json(path: Path) -> Optional[Dict[str, Any]]:
    return load_json(path) if path.exists() else None


def normalize_event(event: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(event)
    if "event_id" not in out and "id" in out:
        out["event_id"] = out["id"]
    return out


def normalize_kg_context(kg_context: Dict[str, Any], event: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(kg_context)
    if "asset_id" not in out:
        asset_ids = ((out.get("seed_context") or {}).get("asset_ids") or [])
        out["asset_id"] = asset_ids[0] if asset_ids else event.get("asset_id")
    if "subgraph_id" not in out:
        out["subgraph_id"] = f"KGCTX::{event['event_id']}::{out.get('asset_id')}"
    return out


def normalize_candidates(cands: Dict[str, Any]) -> Dict[str, Any]:
    """
    Make older 4-dim fixtures compatible with the current engine/schema direction.
    """
    out = dict(cands)

    scoring_cfg = dict(out.get("scoring_config") or {})
    weights = dict(scoring_cfg.get("weights") or {})
    if "telemetry" not in weights:
        # Preserve original intent while making weights sum to 1.0
        weights = {
            "structural": 0.30,
            "temporal": 0.20,
            "telemetry": 0.20,
            "evidence": 0.20,
            "governance": 0.10,
        }
    scoring_cfg["weights"] = weights
    out["scoring_config"] = scoring_cfg

    fixed_candidates = []
    for c in out.get("candidates", []) or []:
        cc = dict(c)
        scores = dict(cc.get("scores") or {})
        if "telemetry" not in scores:
            # conservative backfill for old fixture
            scores["telemetry"] = scores.get("temporal", 0.5)
        cc["scores"] = scores

        if "score_rationale" not in cc:
            cc["score_rationale"] = {
                "structural": "Loaded from test fixture.",
                "temporal": "Loaded from test fixture.",
                "telemetry": "Backfilled for compatibility from temporal score.",
                "evidence": "Loaded from test fixture.",
                "governance": "Loaded from test fixture.",
            }

        if "temporal_evidence" not in cc:
            cc["temporal_evidence"] = {
                "tskr_rule_ids": [],
                "matching_signal_ids": [],
                "window_start": None,
                "window_end": None,
                "relation": "unknown",
                "operator_family": None,
                "mean_lag_hours": None,
                "support": None,
                "pattern_id": None,
            }

        fixed_candidates.append(cc)

    out["candidates"] = fixed_candidates
    return out


def derive_tskr_patterns(
    event: Dict[str, Any],
    telemetry_summary: Dict[str, Any],
    kg_context: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Minimal compatible temporal artifact for fixture-driven dry runs.
    """
    patterns = []
    anomaly_present = any((sig.get("anomalies") or []) for sig in telemetry_summary.get("signals", []) or [])
    for fm in kg_context.get("failure_modes", []) or []:
        fm_id = fm.get("fm_id")
        if not fm_id:
            continue
        patterns.append(
            {
                "pattern_id": f"TSKR::{fm_id}",
                "event_id": event["event_id"],
                "asset_id": event["asset_id"],
                "target_type": "failure_mode",
                "target_id": fm_id,
                "component_id": fm.get("component_id"),
                "relation": "simultaneous" if anomaly_present else "unknown",
                "operator_family": "interval_point",
                "mean_lag_hours": 0.0 if anomaly_present else None,
                "std_lag_hours": None,
                "support": 0.0,
                "confidence": 0.65 if anomaly_present else 0.0,
                "source": "test_rca_orchestrator.py",
            }
        )

    return {
        "event_id": event["event_id"],
        "asset_id": event["asset_id"],
        "patterns": patterns,
        "summary": {
            "has_temporal_support": bool(patterns),
            "mode": "fixture_stub",
            "n_patterns": len(patterns),
        },
        "provenance": {
            "generated_by": "test_rca_orchestrator.py",
            "generated_at": "2026-01-23T10:40:00Z",
        },
    }


def safe_get(d: Optional[Dict[str, Any]], *keys: str, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(k)
    return default if cur is None else cur

## Load and normalize fixtures

Loads all fixture files from `fixtures/case_001_bearing_wear/`. Required files are loaded
directly; optional files return `None` if absent.

Normalization is applied to bring older fixtures into alignment with the current artifact
schemas. The five assertions that follow verify cross-artifact consistency (matching
`event_id` and `asset_id` across all artifacts).

In [ ]:
event = normalize_event(load_json(EXTRACT_DIR / "event.json"))
telemetry_summary = load_json(EXTRACT_DIR / "telemetry_summary.json")
kg_context = normalize_kg_context(load_json(EXTRACT_DIR / "kg_context.json"), event)
causality_candidates = normalize_candidates(load_json(EXTRACT_DIR / "causality_candidates.json"))
evidence_bundle = load_json(EXTRACT_DIR / "evidence_bundle.json")
operational_context = maybe_load_json(EXTRACT_DIR / "operational_context.json")
pm_compliance = maybe_load_json(EXTRACT_DIR / "pm_compliance.json")
rca_card_seed = maybe_load_json(EXTRACT_DIR / "rca_card.json")
tskr_patterns = maybe_load_json(EXTRACT_DIR / "tskr_patterns.json")
if tskr_patterns is None:
    tskr_patterns = derive_tskr_patterns(event, telemetry_summary, kg_context)

assert event["asset_id"] == telemetry_summary["asset_id"]
assert kg_context["event_id"] == event["event_id"]
assert kg_context["asset_id"] == event["asset_id"]
assert causality_candidates.get("event_id") == event["event_id"]
assert evidence_bundle["retrieval_scope"]["asset_id"] == event["asset_id"]

print("Fixture sanity checks passed.")

## Build the orchestrator

Constructs the orchestrator using `build_dev_orchestrator`, which wires together:

- **`RCAArtifactValidator`** — validates each artifact against its JSON schema;
  `VALIDATOR_MODE="compat"` accepts artifacts that are missing optional fields
- **`Neo4jKGContextBuilder`** — wraps the Neo4j client for KG queries
  (not invoked here since `kg_context` is pre-supplied)
- **`TSKRTemporalScorer`** — TSKR Allen-relation temporal scorer
  (not invoked here since `tskr_patterns` is pre-supplied)
- **`CausalityEngineV31`** — baseline causality engine
  (not invoked here since `causality_candidates` is pre-supplied)
- **`EvidenceRetriever`** — document retrieval layer
  (not invoked here since `evidence_bundle` is pre-supplied)
- **`RuleValidatedRCASynthesizerV31`** — RCA card synthesizer (runs live in this test)

The validator will confirm that 11 JSON schemas are loaded from `../schemas/`.
Setting `STOP_ON_VALIDATION_ERROR=False` means schema warnings are logged but do not
abort the run.

In [ ]:
client = Py2Neo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

orchestrator = build_dev_orchestrator(
    output_dir=OUTPUT_DIR,
    client=client,
    database=NEO4J_DATABASE,
    schema_dir=SCHEMA_DIR,
    validator_mode=VALIDATOR_MODE,
    stop_on_validation_error=STOP_ON_VALIDATION_ERROR,
)

print("Orchestrator built.")

In [ ]:
print(type(orchestrator.validator).__name__)
print(getattr(orchestrator.validator, "schema_dir", None))
print(sorted(getattr(orchestrator.validator, "schemas", {}).keys()))

## Run the orchestrator with pre-built artifacts

Calls `orchestrator.run()` passing all pre-built artifacts directly. Because every
intermediate artifact is supplied, the orchestrator skips Stages B–E and runs only:

- **Stage F** — RCA synthesizer: selects the top candidate (`FM_BEARING_WEAR`) and
  generates the RCA card narrative, citations, and recommended actions
- **Stage G** — Finalize manifest: validates the output bundle against all schemas,
  populates `run_manifest`, and evaluates `writeback_ready` / `requires_human_review`

The `try/finally` block ensures the Neo4j client is always closed, even if the run fails.

In [ ]:
try:
    result = orchestrator.run(
        event=event,
        telemetry_summary=telemetry_summary,
        operational_context=operational_context,
        pm_compliance=pm_compliance,
        kg_context=kg_context,
        tskr_patterns=tskr_patterns,
        causality_candidates=causality_candidates,
        evidence_bundle=evidence_bundle,
    )
finally:
    client.close()

print("Run completed.")
print("Returned keys:", sorted(result.keys()))

## Inspect outputs

Prints a JSON dump of each output artifact and then a concise summary dict.

Key fields to check:

| Artifact | What to look for |
|---|---|
| `input_validation` | `ok: true` — all input artifacts pass schema validation |
| `output_validation` | `ok: true` — all output artifacts pass schema validation |
| `run_manifest` | `review_hooks.writeback_ready: true` — result is ready for analyst sign-off |
| `rca_card` | `primary_hypothesis.candidate_id: FM::FM_BEARING_WEAR`, `composite_score: 0.78` |
| `rca_card` | `validation_status.schema_valid: true`, `all_claims_cited: true` |
| `rca_card` | `validation_status.fallback_used: true` — rule-based synthesizer was used (no LLM) |

If either `input_validation.ok` or `output_validation.ok` is `false`, inspect the
`issues` list for schema violations. These typically indicate a fixture drift or a
schema version mismatch.

In [ ]:
for key in [
    "input_validation",
    "output_validation",
    "run_manifest",
    "kg_context",
    "tskr_patterns",
    "causality_candidates",
    "evidence_bundle",
    "rca_card",
]:
    if key in result:
        print(f"\n--- {key} ---")
        print(json.dumps(result[key], indent=2, default=str)[:4000])

summary = {
    "run_id": safe_get(result, "run_context", "run_id"),
    "event_id": safe_get(result, "rca_card", "event_id", default=event["event_id"]),
    "primary_hypothesis": safe_get(result, "rca_card", "primary_hypothesis", default={}),
    "n_candidates": len(safe_get(result, "causality_candidates", "candidates", default=[]) or []),
    "n_evidence": len(safe_get(result, "evidence_bundle", "results", default=[]) or []),
    "input_ok": safe_get(result, "input_validation", "ok"),
    "output_ok": safe_get(result, "output_validation", "ok"),
}

print("\n--- summary ---")
print(json.dumps(summary, indent=2))